# Implement Inference

In [1]:
import os
from datasets import load_dataset
import torch.nn as nn
import torch

from configs.english_german_config import English_german_config
from utils.load_wmt14_en_de_dataset import load_wmt14_en_de, get_training_corpus
from model.bpe_tokenizer import build_and_train_BPE_tokenizer
from model.Transformer import Transformer
from model.generator import Generator
from model.embedding import Embeddings
from model.pos_encoding import PositionalEncoding
from model.training import TrainModel
from model.utils import load_checkpoint
from model.beam_search import BeamSearch

from utils.data_loader import create_data_loaders, filter_ds, pre_tokenize

/Users/tonyavis/miniconda3/envs/AI_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
cfg = English_german_config()
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [4]:
def load_trained_model(cfg:English_german_config, device)-> Transformer:
    """Load a checkpoint, change config for which checkpoint"""

    model = Transformer(cfg=cfg)

    # Tie Weights:💡 From paper: "In our model, we share the same weight matrix between the two embedding layers and the pre-softmax linear transformation, similar to [30]."
    #   - The pre-softmax linear transformation is the Generator, which is the last softmax + linear in the model.
    #   - So, we need to share weights between the scr_embed (Source Embedding), tgt_embed (Target Embedding), and the generator!
    shared_weights = model.src_embed[0].look_up_table.weight
    model.tgt_embed[0].look_up_table.weight = shared_weights
    model.generator.proj.weight = shared_weights
    
    chpt_path = os.path.join(cfg.MODEL_DIR, "checkpoints", cfg.checkpoint_name)

    if not os.path.exists(chpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {chpt_path}")
    
    print(f"Loading weights from {chpt_path}...")
    checkpoint = torch.load(chpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model


In [5]:
tokenizer = build_and_train_BPE_tokenizer(
    cfg=cfg, perc_to_download=cfg.perc_to_download, dataset_iterator=None
)


Loading existing BPE tokenizer from: /Users/tonyavis/Main/AI_projects_and_res/Transformer/model/saved_models/tokenizer/wmt_14_shared_bpe_tokenizer_10_ds_percent.json...


In [6]:
model = load_trained_model(cfg, device)

Loading weights from /Users/tonyavis/Main/AI_projects_and_res/Transformer/model/checkpoints/transformer_epoch_2_10_percent_ds.pt...


In [7]:
def translate_sentence(english_input, model, tokenizer, cfg, device):
    src_encoded = tokenizer.encode(english_input)
    src_tensor = torch.tensor([src_encoded.ids], dtype=torch.long, device=device)

    pad_symbol = cfg.special_tokens["pad_token"]
    src_padding_mask = (src_tensor != pad_symbol).unsqueeze(-2).unsqueeze(-2).to(device)

    max_len = src_tensor.size(1) + 50

    with torch.no_grad():
        pred_seq = BeamSearch(
            model=model,
            src=src_tensor,
            src_padding_mask=src_padding_mask,
            max_len=max_len,
            pad_token_id=0,
            start_token=cfg.special_tokens["sos_token"],
            eos_token=cfg.special_tokens["eos_token"],
            beam_size=4,
            device=device
        )
    
    pred_ids = pred_seq.cpu().numpy().tolist()
    translated_text = tokenizer.decode(pred_ids, skip_special_tokens=True)
    return translated_text

In [ ]:
while True:
    english_input = input("\nEnglish you want to translate to german: ")
    if english_input.lower() in ['quit', 'exit', 'q']:
        break
    german_output = translate_sentence(english_input, model, tokenizer, cfg, device)
    print(f"German: {german_output}")

German: Ich bin gespielt.
German: Ich bin gespielt.
German: Ich möchte mich wählen.
German: Ich bin froh.
German: Ich werde eine Arbeit leisten.
German: Ich möchte auf den Weg eingehen.
German: Ich möchte die Schafe schicken.
German: Ich werde mich anschließen.
German: Die Verzweiflung ist ein Zufall.
German: Das Drama wurde gezogen.
German: Die Klage ist nicht zuhören.
